# Pick and place a cube with Stretch (Isaac Sim, real-robot mode)

The smallest end-to-end manipulation test on this stack: a cube sits on a
pedestal, the Stretch drives up, grasps it, carries it 1.6 m and sets it down on
a second pedestal. Same CRAM-over-giskard path as
`stretch_apartment_cram.ipynb`, with the drawer's articulated kinematics taken
out of the picture.

The point of difference from the drawer demo: **the cube is a real rigid body in
Isaac Sim**, not only an entry in the digital twin. The fingers have to close on
it for real and friction has to hold it through the carry. The sim publishes the
cube's true pose alongside, so after every step you can compare what CRAM
*believes* against what the physics actually did — `cube_status()` below prints
both. A grasp that misses shows up as the two drifting apart.

**Kernel**: select **CRAM**.

## Scene layout

Everything is in the `map` frame (identical to the Isaac world frame here). The
numbers live in `cram_vrb_lab.scenes.props.constants`, and both the Isaac bodies
and their twin counterparts are built from them, so render and twin agree by
construction.

```
        y
        ^
  1.35  |   [base: place] <----- [base: pick] <--        |                                         | approach waypoint
  0.60  |   [place pedestal]     [pick pedestal + cube]
        |
  0.00  |                  * Stretch spawns here
        +---------------------------------------------> x
           -2.6           -2.0    -1.5      -1.0
```

Stretch's arm telescopes out of the base's **right** side — the tool frame
(`link_grasp_center`) sits at `base_link` y = -0.415 retracted and y = -0.935
fully extended, at z = 0.11 + `joint_lift`. So the base parks *alongside* a
pedestal, 0.75 m to its +y, never in front of it.

The same geometry decides how the gripper may come at the cube. At the natural
posture (wrist yaw 0) the tool frame's x-axis — its approach axis — points along
`base_link` -y, straight down the arm, and its z-axis is up. With the base at
yaw 0 that is an approach along map -y, which is CRAM's
`ApproachDirection.LEFT`. The obvious-looking `FRONT` is the wrong one here: it
asks the gripper to point along map +x, which costs a 90° wrist yaw and swings
the grasp centre round to `base_link` (0.248, -0.145), inside the base's own
footprint. The whole-body QP responds by driving the base away from the pedestal
and then goes infeasible. **Grasp along the arm, not across it.**

## Start the simulation and giskard server

Same launcher as the other demos, with `props=True` so the sim spawns the cube
and the two pedestals (`--props`). They are off by default because the pedestals
stand in floor the apartment demos navigate through.

`camera="none"` skips the RTX head camera: nothing here needs it, and the sim
steps faster without it, which the closed-loop controller notices.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))

import os
os.environ["DISPLAY"] = ":0"

from launcher import start_isaac_sim, start_giskard_server, start_rviz, stop

rviz_proc = start_rviz()
sim_proc = start_isaac_sim(camera="none", props=True)
giskard_proc = start_giskard_server()

killed a stale instance of rviz2
starting rviz2, logging to /tmp/rviz.log
starting isaac sim, logging to /tmp/isaac_sim.log
isaac sim ready after 20s
killed a stale instance of giskard server
starting giskard server, logging to /tmp/giskard_server.log
giskard server ready after 12s


## Connect and build a CRAM `Context`

Unchanged from `stretch_apartment_cram.ipynb`: fetch the world the giskard server
built (robot + apartment) over its `fetch_world` service, keep it live with a
`WorldSynchronizer`, and wrap it in a `Context` carrying the Stretch-specific
motion mappings.

In [ ]:
import threading

import nest_asyncio
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()  # CRAM's REAL execution calls GiskardWrapper.execute,
                      # which run_until_completes inside the already-running kernel loop.

from coraplex.datastructures.dataclasses import Context
from coraplex.alternative_motion_mappings.stretch_motion_mapping import (
    StretchMoveToolCenterPoint,
    StretchMoveSim,
    StretchMoveReal,
    StretchClose,
)
from semantic_digital_twin.robots.stretch import Stretch
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

STRETCH_MOTION_MAPPINGS = [
    StretchMoveToolCenterPoint,
    StretchMoveSim,
    StretchMoveReal,
    StretchClose,
]

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_pick_place_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Stretch)
robot = robot[0] if robot else Stretch.from_world(world)

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=STRETCH_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)

KeyboardInterrupt: 

: 

## A small run helper

`with real_robot(...)` sets the execution type to REAL, so `plan.perform()` builds
each action's giskard motion and streams it to the running server.
`collision_avoidance=True` adds an `ExternalCollisionAvoidance` goal against the
apartment (and now the pedestals) the world carries.

In [ ]:
from coraplex.execution_environment import real_robot
from coraplex.plans.factories import sequential, execute_single


def run_plan(plan, collision_avoidance=True):
    """Perform a CRAM plan on the real (sim) robot via giskard."""
    with real_robot(collision_avoidance=collision_avoidance):
        plan.perform()
    print('done')

## 1. Put the props into the digital twin

Isaac already has the cube and pedestals as physics bodies. CRAM plans against
the twin, so the same three boxes have to exist there too —
`add_props_to_twin` builds them from the shared constants. The change goes out
over `/world_sync`, so the giskard server's copy of the world learns about them
and can avoid the pedestals.

`sync_cube_from_sim` then snaps the twin's cube onto the pose Isaac's physics
actually settled it at. Think of it as a one-shot perception stand-in: it is the
only channel through which the twin ever learns the cube is not where CRAM
assumed.

In [4]:
import numpy as np

from cram_vrb_lab.scenes.props import constants as props
from cram_vrb_lab.scenes.props.twin_props import (
    CubePoseSensor,
    add_props_to_twin,
    sync_cube_from_sim,
)

cube = add_props_to_twin(world)
cube_sensor = CubePoseSensor(node)

print('twin cube:', np.round(np.asarray(cube.global_pose.to_np())[:3, 3], 3))
print('sim cube: ', np.round(sync_cube_from_sim(world, cube, cube_sensor), 3))

twin cube: [-1.     0.6    0.055]
sim cube:  [-1.     0.6    0.025]


### Believed vs. actual

The one measurement this notebook is built around. CRAM's `AttachNode` moves the
cube along with the gripper *in the twin* the moment the close-gripper motion
finishes — whether or not the physical fingers caught anything. So the twin
always reports a successful grasp. Isaac does not.

While the cube is held, a gap of a few centimetres is normal (the twin freezes it
at the tool frame; the real cube hangs wherever the fingers gripped it). A gap
that keeps growing means the cube was left behind or dropped.

In [5]:
def cube_status(label=''):
    """Print where CRAM believes the cube is vs where Isaac's physics has it."""
    believed = np.asarray(cube.global_pose.to_np())[:3, 3].ravel()
    actual = np.array(cube_sensor.position())
    print(f'{label:14s} twin {np.round(believed, 3)}  '
          f'sim {np.round(actual, 3)}  gap {np.linalg.norm(believed - actual):.3f} m')
    return believed, actual


cube_status('start')

start          twin [-1.     0.6    0.025]  sim [-1.     0.6    0.025]  gap 0.000 m


(array([-0.99999982,  0.5999999 ,  0.0250001 ]),
 array([-0.99999982,  0.60000008,  0.02500015]))

## 2. Park the arm and open the gripper

Intent-level as ever: CRAM looks up Stretch's parked configuration and the
gripper's open finger position from the semantic model.

**The default open position is too narrow for this cube.** The semantic Stretch
model defines `GripperState.OPEN` as finger angle 0.109 rad
(`semantic_digital_twin/robots/stretch.py`, `StretchGripper.setup_joint_states`),
and measured against the SG3 fingertip collision meshes that parts the pads by
only **3.6 cm** — less than the 5 cm cube. The gripper simply cannot be brought
around it, so the reach ends up pushing the cube instead of enclosing it and the
whole-body QP fights an unreachable grasp pose.

`open_gripper_to` redefines that state on this robot instance, in metres of pad
gap rather than radians. It is a runtime patch of the joint state, so the
vendored model stays untouched and every later `SetGripperAction(OPEN)` — plus
the opening phase inside `PickUpAction` — picks up the new width.

| finger angle | pad gap |
|---|---|
| 0.000 rad (`CLOSE`) | 0.000 m (pads meet) |
| 0.109 rad (default `OPEN`) | 0.036 m |
| 0.277 rad (`CUBE_GRIPPER_GAP`) | 0.090 m |
| 0.600 rad (limit) | 0.191 m |

In [6]:
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction, SetGripperAction
from coraplex.datastructures.enums import Arms
from semantic_digital_twin.datastructures.definitions import GripperState

from cram_vrb_lab.robots.stretch.gripper import measured_pad_gap, open_gripper_to

angle = open_gripper_to(robot, props.CUBE_GRIPPER_GAP)
print(f'GripperState.OPEN is now {angle:.3f} rad '
      f'= {props.CUBE_GRIPPER_GAP:.3f} m between the pads')

run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))

# Commanded is not achieved: the sim articulation carries its own finger limits.
print(f'fingers actually at {measured_pad_gap(world):.3f} m '
      f'(needs to clear the {props.CUBE_SIZE:.3f} m cube)')

GripperState.OPEN is now 0.276 rad = 0.090 m between the pads


[INFO] [1785418961.845559575] [cram_pick_place_node]: giskard/command Goal #0 accepted
[INFO] [1785418969.962609643] [cram_pick_place_node]: giskard/command Goal #0 result received


done


[INFO] [1785418970.347515505] [cram_pick_place_node]: giskard/command Goal #0 accepted


done
fingers actually at 0.000 m (needs to clear the 0.050 m cube)


[INFO] [1785418997.964137212] [cram_pick_place_node]: giskard/command Goal #0 result received


## 3. Drive alongside the pick pedestal

`PICK_BASE_POSITION` is 0.75 m to the cube's +y — the standoff that puts the arm
at roughly two thirds extension. The default (identity) orientation is yaw 0,
which aims the arm's side at the pedestal.

Two legs, each its own `run_plan`: there is no path planning here, the base
drives roughly straight at whatever target it is given, and the straight line
from the spawn to the pick position shaves the pedestal's near corner. Going
west to `APPROACH_WAYPOINT` first and then east along the standing lane clears
both pedestals. (Separate executions per leg are also more reliable than one
merged plan — see the navigation section of `stretch_apartment_cram.ipynb`.)

In [ ]:
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from semantic_digital_twin.spatial_types.spatial_types import Pose
from semantic_digital_twin.spatial_types import Point3


def drive_to(xy):
    target = Pose(Point3.from_iterable([*xy, 0.0]), reference_frame=world.root)
    run_plan(execute_single(NavigateAction(target), context=context))


# drive_to(props.APPROACH_WAYPOINT)
drive_to(props.PICK_BASE_POSITION)

[INFO] [1785419324.979478998] [cram_pick_place_node]: giskard/command Goal #0 accepted


done


[INFO] [1785419354.467229862] [cram_pick_place_node]: giskard/command Goal #0 result received


: 

### Aim the head at the cube

Not needed for the grasp (this demo does not perceive), but it makes the
viewport and RViz show what the robot is working on.

In [9]:
from coraplex.robot_plans.actions.core.navigation import LookAtAction

run_plan(execute_single(LookAtAction(cube.global_pose), context=context))

[INFO] [1785419074.446004535] [cram_pick_place_node]: giskard/command Goal #0 accepted


done


[INFO] [1785419080.116282497] [cram_pick_place_node]: giskard/command Goal #0 result received


## 4. Grasp the cube

`PickUpAction` expands to *open gripper → move to a pre-grasp pose → move to the
grasp pose → close gripper → attach in the twin → lift*. The `GraspDescription`
says how to come at it: `ApproachDirection.LEFT` is the approach that runs along
the arm rather than across it, for the reason spelled out at the top of this
notebook. `VerticalAlignment.NoAlignment` keeps the gripper upright.

`collision_avoidance=False`: the gripper has to get *inside* the avoidance margin
of the cube and the pedestal, which is exactly what the margin forbids.

We re-sync from the sim first, so the grasp is planned against where the cube
really is rather than where it was spawned.

In [16]:
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.datastructures.grasp import GraspDescription
from coraplex.datastructures.enums import ApproachDirection, VerticalAlignment
from coraplex.view_manager import ViewManager

grasp = GraspDescription(
    ApproachDirection.RIGHT,   # approach along map -y = along the arm
    VerticalAlignment.NoAlignment,
    ViewManager.get_end_effector_view(Arms.LEFT, robot),
)

sync_cube_from_sim(world, cube, cube_sensor)
run_plan(
    execute_single(PickUpAction(cube, Arms.LEFT, grasp), context=context),
    collision_avoidance=False,
)
cube_status('after grasp')

[INFO] [1785419300.051246052] [cram_pick_place_node]: giskard/command Goal #0 accepted
[INFO] [1785419310.014646793] [cram_pick_place_node]: giskard/command Goal #0 result received


ExecutionAbortedException: Execution aborted by Giskard.

**If the cube did not come along** (the sim pose is still on the pedestal at
z ≈ 0.725 while the twin pose has risen), the usual causes, in the order worth
checking:

- *The gripper was not open wide enough to go round it.* The first thing to rule
  out, and the reason step 2 prints the achieved pad gap: anything at or below
  the cube's own 5 cm and the fingers hit it on the way in instead of enclosing
  it. Raise `CUBE_GRIPPER_GAP`; if the commanded and achieved numbers disagree,
  the sim articulation's finger limits are the binding ones, not the URDF's.
- *The fingers closed beside the cube.* Watch the Isaac viewport during the
  reach. A lateral miss means the twin's cube pose and the rendered one had
  drifted — re-run the `sync_cube_from_sim` cell and try again.
- *The fingers closed but the cube squirted out.* Raise `CUBE_FRICTION` in
  `cram_vrb_lab/scenes/props/constants.py`, or lower `CUBE_MASS`.
- *The fingers never really closed.* `StretchROS.integrate_joint_velocities`
  clamps a streamed target to `VEL_MAX_LEAD` (0.02 rad) ahead of the measured
  position, so the grip force is `finger stiffness × 0.02`. The finger gains are
  raised in `spawn_stretch`; raise them further if the fingers stall on contact.

## 5. Carry it to the second pedestal

The interesting half: the base drives 1.6 m with the cube in the gripper. Both
standing positions share the same y, so this leg runs down a lane that clears
both pedestals.

The twin carries the cube rigidly attached to the tool frame, so only the sim
pose can tell you whether it survived the drive — watch the gap in
`cube_status()`.

In [10]:
drive_to(props.PLACE_BASE_POSITION)
cube_status('after carry')

[INFO] [1785280808.907451443] [cram_pick_place_node]: giskard/command Goal #0 accepted


done
after carry    twin [-2.734  0.223  0.772]  sim [-0.959  0.476  0.025]  gap 1.943 m


[INFO] [1785280884.381424047] [cram_pick_place_node]: giskard/command Goal #0 result received


(array([-2.73416145,  0.22336232,  0.77235143]),
 array([-0.95850402,  0.47557491,  0.02500016]))

## 6. Put it down

`PlaceAction` is the mirror of the pick: reach the target through the grasp
sequence in reverse, open the gripper, detach in the twin, retract.

In [ ]:
from coraplex.robot_plans.actions.core.placing import PlaceAction

place_target = Pose(
    Point3.from_iterable(props.CUBE_TARGET_POSITION),
    reference_frame=world.root,
)
run_plan(
    execute_single(PlaceAction(cube, place_target, Arms.LEFT), context=context),
    collision_avoidance=False,
)
run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
cube_status('after place')

[INFO] [1785280886.216244300] [cram_pick_place_node]: giskard/command Goal #0 accepted
[INFO] [1785280900.379770598] [cram_pick_place_node]: giskard/command Goal #0 result received


ExecutionAbortedException: Execution aborted by Giskard.

: 

## 7. Verdict

The cube's true resting position against where it was asked to go. Anything under
a few centimetres in x/y and sitting at the pedestal's top height means the whole
chain worked; a z near 0 means it ended up on the floor.

In [ ]:
_, actual = cube_status('final')
target = np.array(props.CUBE_TARGET_POSITION)
error = actual - target
print(f'target {np.round(target, 3)}')
print(f'error  {np.round(error, 3)}   |xy| {np.linalg.norm(error[:2]):.3f} m')
print('on the pedestal' if abs(error[2]) < 0.05 else 'NOT on the pedestal')

## The same thing as one designator

`TransportAction` is the fetch-and-carry composite: navigate into reach, grasp,
lift, carry, place — one designator, with CRAM resolving the standing positions
itself through `reachability_location` instead of the hand-picked
`PICK_BASE_POSITION` / `PLACE_BASE_POSITION` above.

Worth running once the step-by-step version works, since a failure here can come
from either the manipulation or the reachability search. To retry it, restart the
sim so the cube is back on the pick pedestal.

In [ ]:
# from coraplex.robot_plans.actions.composite.transporting import TransportAction
#
# sync_cube_from_sim(world, cube, cube_sensor)
# run_plan(
#     execute_single(TransportAction(cube, place_target, Arms.LEFT), context=context),
#     collision_avoidance=False,
# )
# cube_status('after transport')

## Shutdown

In [ ]:
stop()  # stops the isaac sim + giskard server + rviz started above